In [ ]:
import pandas as pd

import numpy as np
import pandas_ta as ta

import pandas as pd


def load_and_preprocess_data(csv_path: str):
    """
    Loads EURUSD data from CSV and preprocesses it by adding RELATIVE technical features.

    CSV expected columns: [Time, Open, High, Low, Close, Volume]
    The returned DataFrame still contains OHLCV for env internals,
    but `feature_cols` lists only the RELATIVE columns to feed the agent.
    """
    df = pd.read_csv(
        csv_path,
        parse_dates=["Time"],
        dayfirst=True,
    )

    # Strip any trailing spaces in headers (e.g. 'Volume ')
    df.columns = df.columns.str.strip()

    # Datetime index
    df = df.set_index("Time")
    df.sort_index(inplace=True)

    # Ensure numeric
    for col in ["Open", "High", "Low", "Close", "Volume"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Ensure index is datetime if not already
    df["Gmt time"] = pd.to_datetime(df.index)

    # 1. Hour of Day (0-23)
    df["hour"] = df["Gmt time"].dt.hour
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

    # 2. Day of Week (0=Monday, 6=Sunday)
    df["day_of_week"] = df["Gmt time"].dt.dayofweek
    df["day_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["day_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

    # ---- Technicals ----
    # RSI and ATR (already scale-invariant-ish)
    df["rsi_14"] = ta.rsi(df["Close"], length=14)
    df["atr_14"] = ta.atr(df["High"], df["Low"], df["Close"], length=14)

    # Moving averages
    df["ma_20"] = ta.sma(df["Close"], length=20)
    df["ma_50"] = ta.sma(df["Close"], length=50)

    # Slopes of the MAs
    df["ma_20_slope"] = df["ma_20"].diff()
    df["ma_50_slope"] = df["ma_50"].diff()

    # Distance of price from each MA (relative level)
    df["close_ma20_diff"] = df["Close"] - df["ma_20"]
    df["close_ma50_diff"] = df["Close"] - df["ma_50"]

    # MA divergence: MA20 vs MA50
    df["ma_spread"] = df["ma_20"] - df["ma_50"]
    df["ma_spread_slope"] = df["ma_spread"].diff()

    # Drop initial NaNs from indicators
    df.dropna(inplace=True)

    # normalize ["bias_score","confidence","volatility","trend_strength","momentum","skip_flag"]
    for col in [
        "bias_score",
        "confidence",
        "volatility",
        "trend_strength",
        "momentum",
        "skip_flag",
    ]:
        df[col] = (df[col] - df[col].mean()) / df[col].std()

    # Columns the AGENT should see (no raw price levels / raw MAs)
    feature_cols = [
        "hour_sin",
        "hour_cos",
        "day_sin",
        "day_cos",
        "rsi_14",
        "atr_14",
        "ma_20_slope",
        "ma_50_slope",
        "close_ma20_diff",
        "close_ma50_diff",
        "ma_spread",
        "ma_spread_slope",
        "bias_score",
        "confidence",
        "volatility",
        "trend_strength",
        "momentum",
        "skip_flag",
    ]

    return df, feature_cols

In [ ]:
# trading_env.py

from __future__ import annotations

import numpy as np

# Prefer gymnasium if available (SB3 supports it), fallback to gym
try:
    import gymnasium as gym
    from gymnasium import spaces

    _GYMNASIUM = True
except ImportError:
    import gym
    from gym import spaces

    _GYMNASIUM = False


class ForexTradingEnv(gym.Env):
    """
    RL Forex Trading Environment (Position-Persistent)

    Key properties:
      - Observation: rolling window of features + 3 state features (position, time_in_trade, unrealized_pnl_pips)
      - Actions:
          0: HOLD (do nothing)
          1: CLOSE (close position if any)
          2..: OPEN (direction + SL + TP), only effective when flat
      - Position persistence: once open, position remains until:
          - agent sends CLOSE, or
          - SL/TP hit intrabar
      - Friction: spread + commission + optional slippage
      - Reward:
          - realized PnL (pips) minus costs (pips) on closes
          - optional shaping via delta unrealized PnL (pips) while holding
      - Random episode start to reduce memorization / overfit
    """

    metadata = {"render_modes": ["human"]}

    def __init__(
        self,
        df,
        window_size: int = 30,  # what model see each step
        sl_options=None,
        tp_options=None,
        feature_columns=None,  # what we want model to see
        pip_value: float = 0.0001,  # 1 internationally standard EURUSD pip is 0.0001
        spread_pips: float = 0.9,  # it's like a tax on trading
        commission_pips: float = 0.0,  # tax for broker if they allow zero spread, they gain money on this instead
        max_slippage_pips: float = 0.2,  # during news events, the price can slip, 0.2 mean it will randomly add 0.0-0.2 to entry/exit price, which can cause more loss or more profit
        lot_size: float = 100000.0,  # standard account lot size is 100,000
        reward_scale: float = 1.0,  # global reward multiplier to keep numbers in a reasonable range for the agent
        unrealized_delta_weight: float = 0.07,  # give a small reward/penalty for changes in unrealized pips to encourage good holding behavior (can be zero)
        random_start: bool = True,
        min_episode_steps: int = 300,  # minimum steps per episode (for random starts)
        episode_max_steps: int | None = None,  # optional cap (truncation)
        feature_mean: np.ndarray | None = None,  # optional normalization (train-fitted)
        feature_std: np.ndarray | None = None,  # optional normalization (train-fitted)
        allow_flip: bool = False,  # if True, OPEN while in position flips (close+open). Default False.
        hold_reward_weight: float = 0.03,  # small reward per bar for holding a winning position, encourages the agent to realize that "let profits run"
        open_penalty_pips: float = 0.3,  # penalty per open
        time_penalty_pips: float = 0.05,  # penalty if wait too long
    ):
        super().__init__()

        self.df = df.reset_index(drop=True)  # clear index
        self.n_steps = len(
            self.df
        )  # tell the agent to stop when it reaches the end of the data

        # let the agent see only the features we want it to see, but keep all in df for env internals
        if feature_columns is None:
            self.feature_columns = list(self.df.columns)
        else:
            self.feature_columns = list(feature_columns)

        # setting up the tp and sl options for the action space
        if sl_options is None or tp_options is None:
            raise ValueError(
                "sl_options and tp_options must be provided (e.g. [15,20,30])."
            )
        self.sl_options = list(sl_options)
        self.tp_options = list(tp_options)

        # just checking we have enough data for the window size + at least 1 step after
        if self.n_steps <= window_size + 2:
            raise ValueError("Dataframe is too short for the given window_size.")

        self.window_size = int(
            window_size
        )  # 120 for model to see 5 days of H1 data every step it took
        self.pip_value = float(
            pip_value
        )  # depend on the currency pair, for EURUSD it's 0.0001 (1 pip = 0.0001 price move)

        # additional costs
        self.spread_pips = float(spread_pips)  # it's like a tax on trading
        self.commission_pips = float(commission_pips)
        self.max_slippage_pips = float(max_slippage_pips)

        self.lot_size = float(lot_size)
        self.usd_per_pip = self.pip_value * self.lot_size

        # Reward handling
        self.reward_scale = float(reward_scale)  # global reward scaling
        self.unrealized_delta_weight = float(
            unrealized_delta_weight
        )  # reward for changes in unrealized pips while holding
        self.hold_reward_weight = float(
            hold_reward_weight
        )  # rewards the agent simply for staying in a winning position.
        self.open_penalty_pips = float(
            open_penalty_pips
        )  # Only enter if you are very confident
        self.time_penalty_pips = float(
            time_penalty_pips
        )  # don't hold too long, forces the agent to realize that Time is Money

        # Episode handling
        self.random_start = bool(random_start)
        self.min_episode_steps = int(min_episode_steps)
        self.episode_max_steps = (
            episode_max_steps if episode_max_steps is None else int(episode_max_steps)
        )

        # Optional normalization (fit on train only, pass arrays here)
        self.feature_mean = feature_mean
        self.feature_std = feature_std

        self.allow_flip = bool(allow_flip)

        # --- Actions ---
        # 0: HOLD
        # 1: CLOSE
        # 2..: OPEN(direction, sl, tp)
        self.action_map = [("HOLD", None, None, None), ("CLOSE", None, None, None)]
        for direction in [0, 1]:  # 0=short, 1=long
            for sl in self.sl_options:
                for tp in self.tp_options:
                    self.action_map.append(("OPEN", direction, float(sl), float(tp)))

        self.action_space = spaces.Discrete(len(self.action_map))

        # Observation features: df columns + 3 state features
        self.base_num_features = len(self.feature_columns)
        self.state_num_features = 3  # position, time_in_trade, unrealized_pips
        self.num_features = self.base_num_features + self.state_num_features

        self.observation_space = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(self.window_size, self.num_features),
            dtype=np.float32,
        )

        # Internal state
        self._reset_state()

    # ----------------------------
    # Core Helpers
    # ----------------------------

    def _reset_state(self):
        self.current_step = 0
        self.steps_in_episode = 0
        self.terminated = False
        self.truncated = False

        # Position state
        self.position = 0  # 0=flat, +1=long, -1=short
        self.entry_price = None
        self.sl_price = None
        self.tp_price = None
        self.time_in_trade = 0
        self.prev_unrealized_pips = 0.0

        # Accounting
        self.initial_equity_usd = 10000.0
        self.equity_usd = self.initial_equity_usd
        self.balance_usd = (
            self.initial_equity_usd
        )  # added balance to track equity when not in trade

        # Logging
        self.equity_curve = []
        self.last_trade_info = None

    def _get_state_features(self):
        # position in [-1,0,1], time normalized, unrealized in pips (scaled)
        pos = float(self.position)
        t_norm = float(self.time_in_trade) / 1000.0
        unreal_pips = (
            float(self._compute_unrealized_pips()) if self.position != 0 else 0.0
        )
        unreal_scaled = unreal_pips / 100.0  # prevent huge magnitudes
        return np.array([pos, t_norm, unreal_scaled], dtype=np.float32)

    def _compute_unrealized_pips(self):
        if self.position == 0 or self.entry_price is None:
            return 0.0
        close_price = float(self.df.loc[self.current_step, "Close"])
        if self.position == 1:
            pnl_price = close_price - self.entry_price
        else:
            pnl_price = self.entry_price - close_price
        return pnl_price / self.pip_value

    def _apply_optional_normalization(self, obs: np.ndarray) -> np.ndarray:
        if self.feature_mean is None or self.feature_std is None:
            return obs
        mean = self.feature_mean.reshape(1, 1, -1)
        std = self.feature_std.reshape(1, 1, -1)
        std = np.where(std == 0, 1.0, std)
        return (obs - mean) / std

    #! i might change this
    def _get_observation(self):
        start = self.current_step - self.window_size
        if start < 0:
            start = 0

        obs_df = self.df.iloc[start : self.current_step].copy()
        # Get close prices before filtering
        close_prices = obs_df["Close"].values
        # use only selected feature columns for the agent
        obs_df = obs_df[self.feature_columns]

        # If empty (safety), use the first row repeated
        if len(obs_df) == 0:
            base = np.tile(
                self.df.iloc[0][self.feature_columns].values.astype(np.float32),
                (self.window_size, 1),
            )
        else:
            base = obs_df.values.astype(np.float32)
            if base.shape[0] < self.window_size:
                pad_rows = self.window_size - base.shape[0]
                pad = np.tile(base[0], (pad_rows, 1))
                base = np.vstack([pad, base])

        # --- DYNAMIC STATE START ---
        # Instead of getting one value and tiling it, we build arrays for the window
        pos_val = float(self.position)

        if self.position == 0:
            # If flat, the history of the trade in this window is all zeros
            t_series = np.zeros(self.window_size)
            unreal_series = np.zeros(self.window_size)
        else:
            # 1. Dynamic Time: Calculate how 'old' the trade was at each bar in the window
            # np.arange(60) creates [0, 1, ..., 59]
            steps_back = np.arange(self.window_size)
            # We subtract the distance from the 'now' time_in_trade
            t_series = (
                self.time_in_trade - (self.window_size - 1 - steps_back)
            ) / 1000.0
            t_series = np.maximum(
                0, t_series
            )  # Ensure no negative time if window starts before trade

            # 2. Dynamic PnL: Calculate PnL for every Close price in the window
            # close_prices already obtained above
            # Handle padding if the window isn't full yet
            if len(close_prices) < self.window_size:
                pad_len = self.window_size - len(close_prices)
                close_prices = np.concatenate(
                    [
                        np.full(
                            pad_len,
                            (
                                close_prices[0]
                                if len(close_prices) > 0
                                else self.df.iloc[0]["Close"]
                            ),
                        ),
                        close_prices,
                    ]
                )

            if self.position == 1:  # Long
                pips_series = (close_prices - self.entry_price) / self.pip_value
            else:  # Short
                pips_series = (self.entry_price - close_prices) / self.pip_value

            unreal_series = pips_series / 100.0  # Standard scaling

        # Create columns: [Position, Time_History, PnL_History]
        state_block = np.column_stack(
            [np.full(self.window_size, pos_val), t_series, unreal_series]
        )
        # --- DYNAMIC STATE END ---

        # Merge the market features (base) with our new dynamic state_block
        obs = np.hstack([base, state_block]).astype(np.float32)

        # Optional normalization
        obs = self._apply_optional_normalization(obs)

        return obs

    def _sample_slippage_pips(self) -> float:
        if self.max_slippage_pips <= 0:
            return 0.0
        return float(np.random.uniform(0.0, self.max_slippage_pips))

    def _cost_pips_round_trip(self) -> float:
        # Simple friction model (round-trip)
        return self.spread_pips + self.commission_pips

    def _open_position(self, direction: int, sl_pips: float, tp_pips: float):
        # Entry on current close + slippage; costs applied on close (round-trip model)
        close_price = float(self.df.loc[self.current_step, "Close"])
        slip_pips = self._sample_slippage_pips()
        slip_price = slip_pips * self.pip_value

        if direction == 1:  # long
            entry = close_price + slip_price
            sl_price = entry - sl_pips * self.pip_value
            tp_price = entry + tp_pips * self.pip_value
            self.position = 1
        else:  # short
            entry = close_price - slip_price
            sl_price = entry + sl_pips * self.pip_value
            tp_price = entry - tp_pips * self.pip_value
            self.position = -1

        self.entry_price = entry
        self.sl_price = sl_price
        self.tp_price = tp_price
        self.time_in_trade = 0
        self.prev_unrealized_pips = 0.0

        self.last_trade_info = {
            "event": "OPEN",
            "step": self.current_step,
            "position": self.position,
            "entry_price": self.entry_price,
            "sl_price": self.sl_price,
            "tp_price": self.tp_price,
        }

    def _close_position(self, reason: str, exit_price: float):
        # Realized pips
        if self.position == 1:
            pnl_price = exit_price - self.entry_price
        else:
            pnl_price = self.entry_price - exit_price
        realized_pips = pnl_price / self.pip_value

        # Costs in pips (round-trip)
        cost_pips = self._cost_pips_round_trip()
        net_pips = realized_pips - cost_pips

        # Update equity in USD
        self.equity_usd += net_pips * self.usd_per_pip
        self.balance_usd += net_pips * self.usd_per_pip
        self.equity_usd = self.balance_usd

        trade_info = {
            "event": "CLOSE",
            "reason": reason,
            "step": self.current_step,
            "position": self.position,
            "entry_price": self.entry_price,
            "exit_price": exit_price,
            "realized_pips": float(realized_pips),
            "cost_pips": float(cost_pips),
            "net_pips": float(net_pips),
            "equity_usd": float(self.equity_usd),
            "time_in_trade": int(self.time_in_trade),
        }

        # Reset position state
        self.position = 0
        self.entry_price = None
        self.sl_price = None
        self.tp_price = None
        self.time_in_trade = 0
        self.prev_unrealized_pips = 0.0

        self.last_trade_info = trade_info
        return net_pips

    def _check_sl_tp_intrabar_and_maybe_close(self) -> float:
        """
        Checks SL/TP on the *next bar* range [Low, High].
        Conservative rule if both touched: assume SL hits first (worst case).
        Returns realized net pips if closed; otherwise None.
        """
        if self.position == 0:
            return None

        # If last bar, close on close
        if self.current_step >= self.n_steps - 2:
            exit_price = float(self.df.loc[self.current_step, "Close"])
            net_pips = self._close_position("END_OF_DATA", exit_price)
            return net_pips

        next_high = float(self.df.loc[self.current_step + 1, "High"])
        next_low = float(self.df.loc[self.current_step + 1, "Low"])

        if self.position == 1:
            sl_hit = next_low <= self.sl_price
            tp_hit = next_high >= self.tp_price
            if sl_hit and tp_hit:
                # conservative: SL first
                return self._close_position(
                    "SL_AND_TP_SAME_BAR_SL_FIRST", self.sl_price
                )
            elif sl_hit:
                return self._close_position("SL_HIT", self.sl_price)
            elif tp_hit:
                return self._close_position("TP_HIT", self.tp_price)
        else:
            sl_hit = next_high >= self.sl_price
            tp_hit = next_low <= self.tp_price
            if sl_hit and tp_hit:
                return self._close_position(
                    "SL_AND_TP_SAME_BAR_SL_FIRST", self.sl_price
                )
            elif sl_hit:
                return self._close_position("SL_HIT", self.sl_price)
            elif tp_hit:
                return self._close_position("TP_HIT", self.tp_price)

        return None

    # ----------------------------
    # Gym API
    # ----------------------------

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        self._reset_state()

        # Choose start
        if self.random_start:
            max_start = self.n_steps - max(self.min_episode_steps, self.window_size) - 2
            if max_start <= self.window_size:
                self.current_step = self.window_size
            else:
                self.current_step = int(np.random.randint(self.window_size, max_start))
        else:
            self.current_step = self.window_size

        self.steps_in_episode = 0
        self.terminated = False
        self.truncated = False

        obs = self._get_observation()

        if _GYMNASIUM:
            return obs, {}
        return obs

    def step(self, action: int):
        if self.terminated or self.truncated:
            # If someone steps after done, just return current obs with 0 reward
            obs = self._get_observation()
            if _GYMNASIUM:
                return obs, 0.0, True, False, {}
            return obs, 0.0, True, {}

        self.steps_in_episode += 1

        # Reward components
        reward_pips = 0.0
        info = {}

        act_type, direction, sl_pips, tp_pips = self.action_map[int(action)]

        # 1) Apply action logic
        if act_type == "HOLD":
            pass

        elif act_type == "CLOSE":
            if self.position != 0:
                # Close at current close (with slippage)
                close_price = float(self.df.loc[self.current_step, "Close"])
                slip_pips = self._sample_slippage_pips()
                slip_price = slip_pips * self.pip_value
                exit_price = (
                    close_price - slip_price
                    if self.position == 1
                    else close_price + slip_price
                )
                reward_pips += self._close_position("MANUAL_CLOSE", exit_price)

        elif act_type == "OPEN":
            if self.position == 0:
                self._open_position(
                    direction=direction, sl_pips=sl_pips, tp_pips=tp_pips
                )
                # penalty for opening a trade to discourage overtrading
                reward_pips -= self.open_penalty_pips
            else:
                if self.allow_flip:
                    close_price = float(self.df.loc[self.current_step, "Close"])
                    reward_pips += self._close_position("FLIP_CLOSE", close_price)
                    self._open_position(
                        direction=direction, sl_pips=sl_pips, tp_pips=tp_pips
                    )
                    reward_pips -= self.open_penalty_pips

        # 2) If position is open, check SL/TP on next bar intrabar
        realized_now = self._check_sl_tp_intrabar_and_maybe_close()
        if realized_now is not None:
            reward_pips += realized_now

        # 3) If still open, apply reward shaping based on delta-unrealized pips
        if self.position != 0:
            self.time_in_trade += 1

            unreal_now = self._compute_unrealized_pips()

            # --- NEW STOP-OUT LOGIC ---
            self.equity_usd = self.balance_usd + (unreal_now * self.usd_per_pip)
            if self.equity_usd <= 0:
                self.equity_usd = 0.0
                self.terminated = True
                reward_pips -= 5.0
                info["liquidation"] = True
            # --------------------------

            delta_unreal = unreal_now - self.prev_unrealized_pips

            # (a) small bonus for holding a winning trade
            #     proportional to current unrealized profit
            if unreal_now > 0:
                reward_pips += self.hold_reward_weight * unreal_now

            # (b) optional shaping on change in unrealized (can keep small or zero)
            if self.unrealized_delta_weight != 0.0:
                reward_pips += self.unrealized_delta_weight * delta_unreal

            # (c) small time cost per bar in a trade to avoid infinite holding
            reward_pips -= self.time_penalty_pips

            self.prev_unrealized_pips = unreal_now

        # 4) Advance time
        self.current_step += 1

        # 5) Termination / truncation
        if self.current_step >= self.n_steps - 1:
            self.terminated = True

        if (
            self.episode_max_steps is not None
            and self.steps_in_episode >= self.episode_max_steps
        ):
            self.truncated = True

        # 6) Log equity
        self.equity_curve.append(float(self.equity_usd))

        # 7) Build observation
        obs = self._get_observation()

        # 8) Final reward scaling
        reward = float(reward_pips) * self.reward_scale

        # 9) Info
        info.update(
            {
                "equity_usd": float(self.equity_usd),
                "position": int(self.position),
                "time_in_trade": int(self.time_in_trade),
                "reward_pips": float(reward_pips),
                "last_trade_info": self.last_trade_info,
            }
        )

        if _GYMNASIUM:
            return obs, reward, self.terminated, self.truncated, info
        else:
            done = bool(self.terminated or self.truncated)
            return obs, reward, done, info

    def render(self):
        print(
            f"Step={self.current_step} | Equity=${self.equity_usd:,.2f} | "
            f"Pos={self.position} | Entry={self.entry_price} | SL={self.sl_price} | TP={self.tp_price}"
        )

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import CheckpointCallback


def evaluate_model(model: PPO, eval_env: DummyVecEnv, deterministic: bool = True):
    obs = eval_env.reset()

    equity_curve = []
    trade_profits = []
    trade_durations = []

    prev_position = 0
    entry_equity = None

    while True:
        action, _ = model.predict(obs, deterministic=deterministic)
        step_out = eval_env.step(action)

        if len(step_out) == 4:
            obs, rewards, dones, infos = step_out
            done = bool(dones[0])
        else:
            obs, rewards, terminated, truncated, infos = step_out
            done = bool(terminated[0] or truncated[0])

        info = infos[0] if isinstance(infos, (list, tuple)) else infos

        equity = info.get("equity_usd", eval_env.get_attr("equity_usd")[0])
        position = info.get("position", 0)

        equity_curve.append(equity)

        # detect trade open
        if prev_position == 0 and position != 0:
            entry_equity = equity

        # detect trade close
        if prev_position != 0 and position == 0 and entry_equity is not None:
            trade_profit = equity - entry_equity
            trade_profits.append(trade_profit)
            # Add this to track duration
            # You'll need to track 'entry_step' when a trade opens
            duration = eval_env.get_attr("time_in_trade")[0]
            trade_durations.append(duration)
            entry_equity = None

        prev_position = position

        if done:
            break

    final_equity = float(equity_curve[-1])

    # ----- metrics -----
    total_trades = len(trade_profits)

    if total_trades > 0:
        wins = [p for p in trade_profits if p > 0]
        losses = [p for p in trade_profits if p <= 0]

        win_rate = len(wins) / total_trades
        avg_profit = np.mean(wins) if wins else 0
        avg_loss = np.mean(losses) if losses else 0
    else:
        win_rate = 0
        avg_profit = 0
        avg_loss = 0

    # max drawdown
    eq = np.array(equity_curve)
    peak = np.maximum.accumulate(eq)
    drawdown = eq - peak
    max_drawdown = drawdown.min()

    metrics = {
        "total_trades": total_trades,
        "win_rate": win_rate,
        "avg_profit": avg_profit,
        "avg_loss": avg_loss,
        "max_drawdown": max_drawdown,
        "avg_duration": np.mean(trade_durations) if trade_durations else 0,
    }

    return equity_curve, final_equity, metrics


def main():
    file_path = "data/EURUSD_H1_15-16_LLM.csv"
    df, feature_cols = load_and_preprocess_data(file_path)

    # Time split 80/20
    split_idx = int(len(df) * 0.8)
    train_df = df.iloc[:split_idx].copy()
    test_df = df.iloc[split_idx:].copy()

    print("Training bars:", len(train_df))
    print("Testing bars :", len(test_df))

    # ---- Env factories ----
    # Force the AI to only pick trades with at least 1:1.5 or 1:2 R/R
    SL_OPTS = [30, 50, 80]
    TP_OPTS = [60, 100, 160]
    WIN = 120

    # Train env: random starts to reduce memorization
    def make_train_env():
        return ForexTradingEnv(
            df=train_df,
            window_size=WIN,
            sl_options=SL_OPTS,
            tp_options=TP_OPTS,
            spread_pips=1.0,
            commission_pips=0.0,
            max_slippage_pips=0.2,
            random_start=True,
            min_episode_steps=1000,
            episode_max_steps=4000,
            feature_columns=feature_cols,
            hold_reward_weight=0.001,  # 0.05
            open_penalty_pips=0.0,  # 0.5 half a pip per open
            time_penalty_pips=0.005,  # 0.01 pips per bar in trade
            unrealized_delta_weight=0.05,
        )

    # Train-eval env: deterministic start, NO random starts (so curve is stable/reproducible)
    def make_train_eval_env():
        return ForexTradingEnv(
            df=train_df,
            window_size=WIN,
            sl_options=SL_OPTS,
            tp_options=TP_OPTS,
            spread_pips=1.0,
            commission_pips=0.0,
            max_slippage_pips=0.2,
            random_start=False,
            episode_max_steps=None,
            feature_columns=feature_cols,
            hold_reward_weight=0.001,
            open_penalty_pips=0.0,  # half a pip per open
            time_penalty_pips=0.005,  # 0.01 pips per bar in trade
            unrealized_delta_weight=0.05,
        )

    # Test-eval env: deterministic
    def make_test_eval_env():
        return ForexTradingEnv(
            df=test_df,
            window_size=WIN,
            sl_options=SL_OPTS,
            tp_options=TP_OPTS,
            spread_pips=1.0,
            commission_pips=0.0,
            max_slippage_pips=0.2,
            random_start=False,
            episode_max_steps=None,
            feature_columns=feature_cols,
            hold_reward_weight=0.001,
            open_penalty_pips=0.0,  # half a pip per open
            time_penalty_pips=0.005,  # 0.01 pips per bar in trade
            unrealized_delta_weight=0.05,
        )

    train_vec_env = DummyVecEnv([make_train_env])
    train_eval_env = DummyVecEnv([make_train_eval_env])
    test_eval_env = DummyVecEnv([make_test_eval_env])

    # ---- Model ----
    model = PPO(
        policy="MlpPolicy",
        env=train_vec_env,
        verbose=1,
        ent_coef=0.01,  # <--- ADD THIS LINE
        tensorboard_log="./tensorboard_log/",
    )

    # ---- Checkpoints ----
    ckpt_dir = "./checkpoints"
    os.makedirs(ckpt_dir, exist_ok=True)

    checkpoint_callback = CheckpointCallback(
        save_freq=50_000, save_path=ckpt_dir, name_prefix="ppo_eurusd"
    )

    # ---- Train ----
    total_timesteps = 2000000
    model.learn(total_timesteps=total_timesteps, callback=checkpoint_callback)

    # ---- Select best model by OOS final equity ----
    equity_curve_test_last, final_equity_test_last, metrics = evaluate_model(
        model, test_eval_env
    )
    print(f"[OOS Eval] Last model final equity: {final_equity_test_last:.2f}")
    print("Total trades:", metrics["total_trades"])
    print("Win rate:", metrics["win_rate"])
    print("Avg profit:", metrics["avg_profit"])
    print("Avg loss:", metrics["avg_loss"])
    print("Max drawdown:", metrics["max_drawdown"])

    ckpts = sorted(
        [
            f
            for f in os.listdir(ckpt_dir)
            if f.endswith(".zip") and f.startswith("ppo_eurusd")
        ],
        key=lambda x: os.path.getmtime(os.path.join(ckpt_dir, x)),
    )

    # Now the loop will work
    best_score = -np.inf
    best_path = None

    for ck in ckpts:
        ck_path = os.path.join(ckpt_dir, ck)
        try:
            m = PPO.load(ck_path, env=test_eval_env)
            equity_curve, final_eq, metrics = evaluate_model(m, test_eval_env)

            # --- NEW RISK-ADJUSTED CALCULATION ---
            # We calculate a 'Score'. If max_drawdown is 0, we avoid division by zero.
            dd = abs(metrics["max_drawdown"])
            if dd < 1.0:
                dd = 1.0

            # Score = Total Profit / Max Drawdown
            # This is a simplified 'Calmar Ratio'. High score = high efficiency.
            profit = final_eq - 10000.0
            current_score = profit / dd

            print(
                f"[OOS Eval] {ck} -> Equity: {final_eq:.2f}, DD: {metrics['max_drawdown']:.2f}, Score: {current_score:.4f}"
            )

            if current_score > best_score:
                best_score = current_score
                best_path = ck_path
            # -------------------------------------

        except Exception as e:
            print(f"[Skip] Could not evaluate checkpoint {ck}: {e}")

    # Decide best model
    if best_path is None:
        print("Using last model as best (Fallback).")
        best_model = model
    else:
        print(f"Using best checkpoint by Score: {best_path} (Score: {best_score:.4f})")
        best_model = PPO.load(best_path, env=train_vec_env)

    best_model.save("model_eurusd_best")
    print("Best model saved: model_eurusd_best")

    # ---- Plot BOTH: in-sample vs out-of-sample ----
    equity_curve_train, final_equity_train, metrics_train = evaluate_model(
        best_model, train_eval_env
    )
    equity_curve_test, final_equity_test, metrics_test = evaluate_model(
        best_model, test_eval_env
    )

    print(f"[IS Eval]  Final equity (train): {final_equity_train:.2f}")
    print("Total trades:", metrics_train["total_trades"])
    print("Win rate:", metrics_train["win_rate"])
    print("Avg profit:", metrics_train["avg_profit"])
    print("Avg loss:", metrics_train["avg_loss"])
    print("Max drawdown:", metrics_train["max_drawdown"])

    print(f"[OOS Eval] Final equity (test) : {final_equity_test:.2f}")
    print("Total trades:", metrics_test["total_trades"])
    print("Win rate:", metrics_test["win_rate"])
    print("Avg profit:", metrics_test["avg_profit"])
    print("Avg loss:", metrics_test["avg_loss"])
    print("Max drawdown:", metrics_test["max_drawdown"])

    plt.figure(figsize=(12, 6))
    plt.plot(equity_curve_train, label="Train (in-sample) equity")
    plt.plot(equity_curve_test, label="Test (out-of-sample) equity")
    plt.title("Equity Curves: In-sample vs Out-of-sample (Best Model)")
    plt.xlabel("Steps")
    plt.ylabel("Equity ($)")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
main()